In [2]:
import sys
import requests
from bs4 import BeautifulSoup
import re
import unicodedata
import pandas as pd

# --- CÁC HÀM PHỤ TRỢ ĐỂ XỬ LÝ TEXT KHI CÀO TỪ WIKIPEDIA ---
def date_time(table_cells):
    """Trả về ngày và giờ từ ô bảng HTML"""
    return [data.strip() for data in list(table_cells.strings)][0:2]

def booster_version(table_cells):
    """Trả về phiên bản tên lửa (Booster Version)"""
    out=[i for i in table_cells.strings if i.strip() != '']
    if len(out) > 0:
        return out[0].strip()
    return None

def landing_status(table_cells):
    """Trả về trạng thái hạ cánh"""
    out=[i for i in table_cells.strings]
    if len(out) > 0:
        return out[0].strip()
    return None

def get_mass(table_cells):
    """Trích xuất khối lượng payload (kg)"""
    mass=unicodedata.normalize("NFKD", table_cells.text).strip()
    if mass:
        mass.find("kg")
        new_mass=mass[0:mass.find("kg")+2]
    else:
        new_mass=0
    return new_mass

def extract_column_from_header(row):
    """Trích xuất tên cột từ header của bảng HTML"""
    if (row.br):
        row.br.extract()
    if row.a:
        row.a.extract()
    if row.sup:
        row.sup.extract()
    colunm_name = ' '.join(row.contents)
    if not(colunm_name.strip().isdigit()):
        colunm_name = colunm_name.strip()
        return colunm_name

# --- BẮT ĐẦU CÀO DỮ LIỆU ---

# Sử dụng URL tĩnh (snapshot ngày 9/6/2021) của Wikipedia để cấu trúc HTML không bị thay đổi
static_url = "https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922"

# === BƯỚC KHẮC PHỤC LỖI: Thêm User-Agent để Wikipedia không chặn ===
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

# Gửi yêu cầu HTTP GET với headers
response = requests.get(static_url, headers=headers)
html_content = response.text

# Tạo đối tượng BeautifulSoup
soup = BeautifulSoup(html_content, 'html.parser')

if soup.title is not None:
    print("Tiêu đề trang:", soup.title.string) # Xác nhận đã kết nối thành công
else:
    print("Không tìm thấy tiêu đề, vui lòng kiểm tra lại kết nối mạng!")

# Tìm tất cả các bảng trên trang
html_tables = soup.find_all('table')

# Khởi tạo một dictionary để lưu dữ liệu cào được
launch_dict = {
    'Flight No.': [],
    'Launch site': [],
    'Payload': [],
    'Payload mass': [],
    'Orbit': [],
    'Customer': [],
    'Launch outcome': [],
    'Version Booster': [],
    'Booster landing': [],
    'Date': [],
    'Time': []
}

# Lọc và cào dữ liệu từ các bảng chứa thông tin phóng (Bảng từ thứ 3 đến thứ 11)
extracted_row = 0
for table_number in range(2, 10):
    table = html_tables[table_number]
    for rows in table.find_all("tr"):
        if rows.th:
            if rows.th.string:
                flight_number = rows.th.string.strip()
                flag = flight_number.isdigit()
        else:
            flag = False
            
        if flag:
            extracted_row += 1
            row_cells = rows.find_all('td')
            
            # 1. Số thứ tự chuyến bay
            launch_dict['Flight No.'].append(flight_number)
            
            # 2. Ngày và Giờ phóng
            datatimelist = date_time(row_cells[0])
            date = datatimelist[0].strip(',')
            time = datatimelist[1]
            launch_dict['Date'].append(date)
            launch_dict['Time'].append(time)
            
            # 3. Phiên bản Booster (Tên lửa)
            bv = booster_version(row_cells[1])
            if not bv:
                bv = row_cells[1].a.string if row_cells[1].a else None
            launch_dict['Version Booster'].append(bv)
            
            # 4. Địa điểm phóng (Launch Site)
            launch_site = row_cells[2].a.string if row_cells[2].a else None
            launch_dict['Launch site'].append(launch_site)
            
            # 5. Tên hàng hóa (Payload)
            payload = row_cells[3].a.string if row_cells[3].a else None
            launch_dict['Payload'].append(payload)
            
            # 6. Khối lượng hàng hóa (Payload Mass)
            payload_mass = get_mass(row_cells[4])
            launch_dict['Payload mass'].append(payload_mass)
            
            # 7. Quỹ đạo (Orbit)
            orbit = row_cells[5].a.string if row_cells[5].a else None
            launch_dict['Orbit'].append(orbit)
            
            # 8. Khách hàng (Customer)
            customer = row_cells[6].a.string if (row_cells[6] and row_cells[6].a) else None
            launch_dict['Customer'].append(customer)
            
            # 9. Kết quả phóng (Launch Outcome)
            launch_outcome = list(row_cells[7].strings)[0].strip() if row_cells[7] else None
            launch_dict['Launch outcome'].append(launch_outcome)
            
            # 10. Trạng thái hạ cánh Booster (Booster Landing)
            booster_landing = landing_status(row_cells[8])
            launch_dict['Booster landing'].append(booster_landing)

# Chuyển đổi sang DataFrame
df_scraped = pd.DataFrame(launch_dict)

# Lưu thành file CSV
df_scraped.to_csv('spacex_web_scraped.csv', index=False)
print(f"Cào dữ liệu hoàn tất! Đã trích xuất {extracted_row} dòng dữ liệu.")
print("File 'spacex_web_scraped.csv' đã được tạo thành công.")

Tiêu đề trang: List of Falcon 9 and Falcon Heavy launches - Wikipedia
Cào dữ liệu hoàn tất! Đã trích xuất 103 dòng dữ liệu.
File 'spacex_web_scraped.csv' đã được tạo thành công.
